In [1]:
from pathlib import Path
import numpy as np

def load_grain_Fp_file(file_path):
    with open(file_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    # first 10 lines are headers
    headers = lines[:10]  # inc, Fp11, ..., Fp33

    data = {}
    i = 10

    while i < len(lines):
        label = lines[i]              # e.g. "increment_40"
        values = lines[i+1:i+10]      # next 9 values

        Fp = np.array(values, dtype=float).reshape(3, 3)

        inc = int(label.split("_")[1])
        data[inc] = Fp

        i += 10

    return data

def load_grain_strain_file(file_path):
    with open(file_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    # first 10 lines are headers
    headers = lines[:10]  # inc, eps_11, ..., eps_33

    data = {}
    i = 10

    while i < len(lines):
        label = lines[i]              # e.g. "increment_20"
        values = lines[i+1:i+10]      # next 9 numbers

        strain = np.array(values, dtype=float).reshape(3, 3)

        # store by increment label or number
        inc = int(label.split("_")[1])
        data[inc] = strain

        i += 10

    return data

def load_grain_stress_file(file_path):
    with open(file_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    # first 10 lines are headers
    data = {}
    i = 10

    while i < len(lines):
        label = lines[i]              # e.g. "increment_40"
        values = lines[i + 1 : i + 10]

        if len(values) != 9:
            raise ValueError(f"Incomplete stress block at {label} in {file_path}")

        sigma = np.array(values, dtype=float).reshape(3, 3)

        inc = int(label.split("_")[1])
        data[inc] = sigma

        i += 10

    return data

# def build_feature_tensor(
#     all_grain_quaternions,
#     all_grain_Fp,
#     all_grain_strain,
#     all_grain_stress,
#     n_grains=100,
# ):
#     # Use increment list from grain 0 (assumes consistent across grains)
#     incs = sorted(all_grain_quaternions[0].keys())  # e.g. [0,10,...,120]
#     n_incs = len(incs)
#     n_feat = 4 + 9 + 9 + 9  # quat + Fp + strain + stress = 31

#     X = np.zeros((n_grains, n_incs, n_feat), dtype=np.float32)

#     for g in range(n_grains):
#         for t, inc in enumerate(incs):
#             # sanity checks (optional but recommended)
#             if inc not in all_grain_quaternions[g]:
#                 raise KeyError(f"Missing quaternion: grain {g}, inc {inc}")
#             if inc not in all_grain_Fp[g]:
#                 raise KeyError(f"Missing Fp: grain {g}, inc {inc}")
#             if inc not in all_grain_strain[g]:
#                 raise KeyError(f"Missing strain: grain {g}, inc {inc}")
#             if inc not in all_grain_stress[g]:
#                 raise KeyError(f"Missing stress: grain {g}, inc {inc}")

#             q = all_grain_quaternions[g][inc].reshape(4)          # (4,)
#             Fp = all_grain_Fp[g][inc].reshape(9)                  # (9,)
#             eps = all_grain_strain[g][inc].reshape(9)             # (9,)
#             sig = all_grain_stress[g][inc].reshape(9)             # (9,)

#             X[g, t, :] = np.concatenate([q, Fp, eps, sig])

#     return X

In [2]:
def load_grain_strain_permatpoint_file(file_path):
    """
    Loads per-material-point strain data for one grain.
    Handles bracket arrays that may span multiple lines.

    Returns:
      data[inc] -> (n_mp, 3, 3)
    """
    with open(file_path, "r") as f:
        lines = [line.rstrip("\n") for line in f if line.strip()]

    # First 10 non-empty lines are headers (as in your files)
    if len(lines) < 10:
        raise ValueError(f"{file_path}: too short to contain headers.")
    i = 10

    def read_bracket_array(lines, start_idx):
        """
        Reads a numpy-style bracket array that may span multiple lines.
        Returns (array_1d, next_idx).
        """
        s = lines[start_idx].strip()

        if "[" not in s:
            raise ValueError(f"{file_path}: expected '[' at line {start_idx}, got: {s[:80]}")

        buf = [s]
        j = start_idx
        # Keep consuming lines until we see a closing ']'
        while "]" not in buf[-1]:
            j += 1
            if j >= len(lines):
                raise ValueError(f"{file_path}: unterminated bracket array starting at line {start_idx}")
            buf.append(lines[j].strip())

        joined = " ".join(buf)
        # Extract content inside the outermost [ ... ]
        left = joined.find("[")
        right = joined.rfind("]")
        content = joined[left+1:right].strip()

        arr = np.fromstring(content, sep=" ", dtype=float)
        return arr, j + 1  # next index after the array

    data = {}

    while i < len(lines):
        label = lines[i].strip()  # e.g. "increment_20"
        if not label.startswith("increment_"):
            raise ValueError(f"{file_path}: expected 'increment_*' at line {i}, got '{label}'")

        inc = int(label.split("_")[1])
        i += 1

        # Read 9 bracket arrays (eps_11 ... eps_33)
        arrays = []
        for _ in range(9):
            arr, i = read_bracket_array(lines, i)
            arrays.append(arr)

        n_mp = arrays[0].shape[0]
        if any(a.shape[0] != n_mp for a in arrays):
            raise ValueError(f"{file_path}: inconsistent #matpoints in increment_{inc}")

        # arrays: list of (n_mp,) -> stack -> (9, n_mp) -> transpose -> (n_mp, 9)
        E9 = np.stack(arrays, axis=0).T  # (n_mp, 9)

        # reshape to (n_mp, 3, 3) with order:
        # 11,12,13,21,22,23,31,32,33
        # data[inc] = E9.reshape(n_mp, 3, 3)
        data[inc] = E9

    return data


def build_feature_tensor(
    all_grain_quaternions,
    all_grain_Fp,
    all_grain_strain,
    all_grain_stress,
    rve_dE9,                 # <-- NEW: (n_incs, 9)
    n_grains=100,
):
    incs = sorted(all_grain_quaternions[0].keys())
    n_incs = len(incs)

    if rve_dE9.shape != (n_incs, 9):
        raise ValueError(f"rve_dE9 shape mismatch: got {rve_dE9.shape}, expected {(n_incs, 9)}")

    n_feat = 4 + 9 + 9 + 9 + 9  # quat + Fp + strain + stress + RVE_dE9 = 40
    X = np.zeros((n_grains, n_incs, n_feat), dtype=np.float32)

    for g in range(n_grains):
        for t, inc in enumerate(incs):
            q  = all_grain_quaternions[g][inc].reshape(4).astype(np.float32)
            Fp = all_grain_Fp[g][inc].reshape(9).astype(np.float32)
            eps = all_grain_strain[g][inc].reshape(9).astype(np.float32)
            sig = all_grain_stress[g][inc].reshape(9).astype(np.float32)

            rve = rve_dE9[t]  # already float32, shape (9,)

            X[g, t, :] = np.concatenate([q, Fp, eps, sig, rve], axis=0)

    return X

In [3]:
# Project root (GNN-for-Grain-Track)
PROJECT_ROOT = Path.cwd().parent

load_types = ["Compression" , "Traction"]
directions = ["X", "XZ", "Z"] 

X_list = []
loadcases = []  # keep ordering

for load_type in load_types:
    for direction in directions:
        loadcase = f"{load_type}{direction}"
        loadcases.append(loadcase)
        folder = (PROJECT_ROOT / "2D_10%strain_100grains" / "results_grain_matpoint" / loadcase)
        
        all_grain_Fp = {} # grains -> increments -> Fp
        all_grain_quaternions = {}
        all_grain_strain = {}
        all_grain_stress = {}

        for i in range(0, 100):
            # load grain Fp averages
            file_path = folder / "for_ML_Fp_average" / f"grain_{i}.txt"
            if not file_path.exists():
                raise ReferenceError(f'File: {file_path} does not exist')
            all_grain_Fp[i] = load_grain_Fp_file(file_path)

            # load grain quaternion averages
            file_path = folder / "for_ML_quaternion_average" / f"grain_{i}_avg_quaternion.txt"
            if not file_path.exists():
                raise ReferenceError(f'File: {file_path} does not exist')
            arr = np.loadtxt(file_path, comments="#")  # columns: inc, q0, q1, q2, q3
            grain_dict = {}
            for row in arr:
                inc = int(row[0])
                q = row[1:5].astype(float)   # shape (4,)
                grain_dict[inc] = q
            all_grain_quaternions[i] = grain_dict

            # load grain strain averages
            file_path = folder / "for_ML_strain_average" / f"grain_{i}.txt"
            if not file_path.exists():
                raise ReferenceError(f'File: {file_path} does not exist')
            all_grain_strain[i] = load_grain_strain_file(file_path) # loads 3x3 tensor for 9 strain components

            # load grain stress averages
            file_path = folder / "for_ML_stress_average" / f"grain_{i}.txt"
            if not file_path.exists():
                raise ReferenceError(f'File: {file_path} does not exist')
            all_grain_stress[i] = load_grain_stress_file(file_path)


        # ---------- RVE Calculation (ONCE per loadcase) ----------
        grain_strain = []  # length 100, each entry: inc -> (n_mp,3,3)
        for i in range(100):
            file_path = folder / "for_ML_strain_perMatpoint_pergrain" / f"grain_{i}.txt"
            if not file_path.exists():
                raise ReferenceError(f"File: {file_path} does not exist")
            grain_strain.append(load_grain_strain_permatpoint_file(file_path))

        incs = sorted(grain_strain[0].keys())

        # accumulated RVE strain per increment (3,3)
        E_rve = {}
        for inc in incs:
            all_mp = [grain_strain[g][inc] for g in range(100)]  # list of (n_mp_g,3,3)
            all_mp = np.concatenate(all_mp, axis=0)              # (total_mp,3,3)
            E_rve[inc] = all_mp.mean(axis=0)                     # (3,3)

        # incremental Δε as 9-vector (13,9)
        dE9 = np.zeros((len(incs), 9), dtype=np.float32)
        for t in range(1, len(incs)):
            dE = E_rve[incs[t]] - E_rve[incs[t-1]]              # (3,3)
            dE9[t] = dE.reshape(9).astype(np.float32)           # (9,)


        # (100 grains, 13 increments, 40 features (4 quaternions + 9 Fp + 9 strain + 9 stress + 9 RVE strain average))
        X = build_feature_tensor(
            all_grain_quaternions,
            all_grain_Fp,
            all_grain_strain,
            all_grain_stress,
            dE9,
            n_grains=100
        )
        X_list.append(X)

X_all = np.stack(X_list, axis=0) # (6 loadcases, 100 grains, 13 increments, 40 features)

In [23]:
# print(X_all.shape)
print(X_all[0, 1, 1, 22:31])

[-9.5256520e+07 -8.0861755e+06  3.5594695e+06 -8.0861755e+06
  1.3624875e+06  7.9278285e+06  3.5594695e+06  7.9278285e+06
 -2.3516492e+06]


Save as numpy

In [31]:
save_path = Path("simulation_grain_data/grain_feature_tensor.npy")

np.save(save_path, X_all)

Calculating Avg RVE Accumulated Strain

In [5]:
import numpy as np

def load_grain_strain_permatpoint_file(file_path):
    """
    Loads per-material-point strain data for one grain.
    Handles bracket arrays that may span multiple lines.

    Returns:
      data[inc] -> (n_mp, 3, 3)
    """
    with open(file_path, "r") as f:
        lines = [line.rstrip("\n") for line in f if line.strip()]

    # First 10 non-empty lines are headers (as in your files)
    if len(lines) < 10:
        raise ValueError(f"{file_path}: too short to contain headers.")
    i = 10

    def read_bracket_array(lines, start_idx):
        """
        Reads a numpy-style bracket array that may span multiple lines.
        Returns (array_1d, next_idx).
        """
        s = lines[start_idx].strip()

        if "[" not in s:
            raise ValueError(f"{file_path}: expected '[' at line {start_idx}, got: {s[:80]}")

        buf = [s]
        j = start_idx
        # Keep consuming lines until we see a closing ']'
        while "]" not in buf[-1]:
            j += 1
            if j >= len(lines):
                raise ValueError(f"{file_path}: unterminated bracket array starting at line {start_idx}")
            buf.append(lines[j].strip())

        joined = " ".join(buf)
        # Extract content inside the outermost [ ... ]
        left = joined.find("[")
        right = joined.rfind("]")
        content = joined[left+1:right].strip()

        arr = np.fromstring(content, sep=" ", dtype=float)
        return arr, j + 1  # next index after the array

    data = {}

    while i < len(lines):
        label = lines[i].strip()  # e.g. "increment_20"
        if not label.startswith("increment_"):
            raise ValueError(f"{file_path}: expected 'increment_*' at line {i}, got '{label}'")

        inc = int(label.split("_")[1])
        i += 1

        # Read 9 bracket arrays (eps_11 ... eps_33)
        arrays = []
        for _ in range(9):
            arr, i = read_bracket_array(lines, i)
            arrays.append(arr)

        n_mp = arrays[0].shape[0]
        if any(a.shape[0] != n_mp for a in arrays):
            raise ValueError(f"{file_path}: inconsistent #matpoints in increment_{inc}")

        # arrays: list of (n_mp,) -> stack -> (9, n_mp) -> transpose -> (n_mp, 9)
        E9 = np.stack(arrays, axis=0).T  # (n_mp, 9)

        # reshape to (n_mp, 3, 3) with order:
        # 11,12,13,21,22,23,31,32,33
        # data[inc] = E9.reshape(n_mp, 3, 3)
        data[inc] = E9

    return data


In [6]:
# Project root (GNN-for-Grain-Track)
PROJECT_ROOT = Path.cwd().parent

load_types = ["Compression" , "Traction"]
directions = ["X", "XZ", "Z"] 

X_list = []
loadcases = []  # keep ordering

rve_strain_all = {}  # loadcase -> dict[inc] -> (3,3)

for load_type in load_types:
    for direction in directions:
        loadcase = f"{load_type}{direction}"
        loadcases.append(loadcase)
        folder = (PROJECT_ROOT / "2D_10%strain_100grains" / "results_grain_matpoint" / loadcase)

        grain_strain = []  # list of dicts: grain_strain[g][inc] -> (n_mp,3,3)

        for i in range(0, 100):
            file_path = folder / "for_ML_strain_perMatpoint_pergrain" / f"grain_{i}.txt"
            if not file_path.exists():
                raise ReferenceError(f'File: {file_path} does not exist')
            
            grain_data = load_grain_strain_permatpoint_file(file_path)
            grain_strain.append(grain_data)

        # --------------------------------------------------
        # 2) Compute RVE mean strain per increment
        # --------------------------------------------------
        incs = sorted(grain_strain[0].keys())

        E_rve = {}  # inc -> (3,3)

        for inc in incs:
            # collect all matpoints from all grains
            all_mp = []
            for g in range(100):
                all_mp.append(grain_strain[g][inc])  # (n_mp_g,3,3)

            all_mp = np.concatenate(all_mp, axis=0)  # (total_mp,3,3)
            E_rve[inc] = all_mp.mean(axis=0)         # (3,3)

        rve_strain_all[loadcase] = E_rve

In [11]:
print(rve_strain_all)

for item in list(rve_strain_all['CompressionZ'].values()):
    print(item)

{'CompressionX': {0: array([ 2.09673543e-17, -3.66139073e-18, -8.93549254e-19, -3.66139073e-18,
       -3.86221027e-17, -9.85115309e-18, -8.93549254e-19, -9.85115309e-18,
       -8.63533290e-18]), 10: array([ 2.16786403e-04, -4.15780583e-05,  1.78772865e-04, -4.15780583e-05,
       -2.35249999e-04, -2.80979435e-05,  1.78772865e-04, -2.80979435e-05,
        1.84635460e-05]), 20: array([ 5.20605294e-04, -9.36667441e-05,  3.71699543e-04, -9.36667441e-05,
       -5.07651929e-04, -2.37025976e-05,  3.71699543e-04, -2.37025976e-05,
       -1.29534034e-05]), 30: array([ 8.27073160e-04, -1.49830312e-04,  5.63388981e-04, -1.49830312e-04,
       -7.83709170e-04, -7.32727340e-06,  5.63388981e-04, -7.32727340e-06,
       -4.33639970e-05]), 40: array([ 1.13532606e-03, -2.09876600e-04,  7.49877241e-04, -2.09876600e-04,
       -1.06345143e-03,  1.86099407e-05,  7.49877241e-04,  1.86099407e-05,
       -7.18744159e-05]), 50: array([ 1.44482293e-03, -2.72554247e-04,  9.29964825e-04, -2.72554247e-04,
    